# 05 — Contraste INE vs ISTAC

Complemento al cubo ISTAC: pernoctaciones mensuales de la Encuesta de
Ocupación Hotelera (INE, tabla Tempus3 **67190**) para nacional, Canarias
CCAA y provincias Las Palmas / Santa Cruz de Tenerife.

Pregunta: ¿la demanda provincial (pernoctaciones INE) se mueve al unísono
con el RevPAR agregado por isla del ISTAC, o hay desacoplamientos?

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from src.data import load_harmonized_series
from src.ine import save_contrast_series

ine_path = save_contrast_series()
ine = pd.read_csv(ine_path, parse_dates=["fecha"])
istac = load_harmonized_series()
print(f"INE: {ine['serie'].nunique()} series desde {ine['fecha'].min().date()}")
print(f"ISTAC: {istac['isla'].nunique()} islas desde {istac['fecha'].min().date()}")

In [ ]:
PROVINCE_ISLANDS = {
    "las_palmas": ["Gran Canaria", "Lanzarote", "Fuerteventura"],
    "tenerife": ["Tenerife", "La Palma", "La Gomera", "El Hierro"],
}

revpar_prov = {
    prov: istac[istac["isla"].isin(islands)].groupby("fecha")["revpar_eur"].sum()
    for prov, islands in PROVINCE_ISLANDS.items()
}
revpar_df = pd.DataFrame(revpar_prov)
pernoct = ine.pivot_table(index="fecha", columns="serie", values="valor", aggfunc="sum")

fig, axes = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
pernoct[["nacional_pernoctaciones", "canarias_pernoctaciones"]].plot(ax=axes[0], linewidth=1.8)
axes[0].set_title("Pernoctaciones hoteleras INE (nacional vs Canarias)")
axes[0].set_ylabel("Pernoctaciones")
axes[0].axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2021-06-01"), color="grey", alpha=0.15)

revpar_df.plot(ax=axes[1], linewidth=1.8)
axes[1].set_title("RevPAR mensual ISTAC agregado por provincia (suma de islas)")
axes[1].set_ylabel("€ RevPAR")
axes[1].axvspan(pd.Timestamp("2020-03-01"), pd.Timestamp("2021-06-01"), color="grey", alpha=0.15)

plt.tight_layout()
Path("reports/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("reports/figures/05_ine_contrast_canarias.png", dpi=120, bbox_inches="tight")
plt.show()